### **Agente Langgraph com Gravação em Store de Longterm**
#### Marcio Morelli Soares - RM371246 1AIER

In [ ]:
!pip install -U langgraph -q
!pip install -U langgraph-checkpoint-sqlite -q
!pip install langchain-openai
import uuid
import sqlite3
from langgraph.store.sqlite import SqliteStore
import os
from dotenv import load_dotenv
from pathlib import Path
dotenv_path =Path("../../.env")
load_dotenv(dotenv_path=dotenv_path, override=True)


In [ ]:

# Verifica se a variável de ambiente OPENAI_API_KEY está definida

api_key = os.getenv("OPENAI_API_KEY", "").strip()
 
if not api_key:
    raise RuntimeError("OPENAI_API_KEY não foi encontrado no arquivo .env.")
 
if not api_key.startswith("sk-"):
    raise RuntimeError("OPENAI_API_KEY não é válido. Certifique-se de que a chave começa com 'sk-'.")

print("Chave de API encontrada")

In [ ]:
#Cria um BD e sua conexão
conn = sqlite3.connect("dbShoppingCart.db", check_same_thread=False)

semanticDB = SqliteStore(conn, index={"embed": "openai:text-embedding-3-small", "dims": 1536})
semanticDB.setup()
semanticDB.conn.commit()


In [ ]:
# Criando 2 Namespaces distintos para armazenar os dados de histórico de compras 
# e preferências dos usuários

ns_compras = ("historico_compras",)
ns_preferencias = ("preferencias",)


In [ ]:
# Criando listas de pedidos e preferências para o Cliente 1 e o Cliente 2

pedido_1_client01 = ["IPhone 14 Pro Max", "Carregador MagSafe compativel IPhone", "Capa de Silicone IPhone 14 Pro Max"]
pedido_2_client01 = ["Fone de Ouvido AirPods Pro", "Cabo USB-C para Lightning", "Suporte Veicular para IPhone"]
pedido_3_client01 = ["Apple Watch Series 8", "Pulseira Esportiva para Apple Watch", "Carregador Magnético Apple Watch"]
pedido_4_client01 = ["MacBook Pro 16", "Adaptador USB-C para HDMI", "Bolsa para MacBook Pro"]

preferencia_1_client01 = "Cliente 01 prefere produtos da Apple e valoriza a qualidade e durabilidade dos dispositivos."
preferencia_2_client01 = "Tecnologia é uma prioridade para o Cliente 01, e ele está disposto a investir em produtos premium que ofereçam desempenho e inovação."
preferencia_3_client01 = "Monitoramento de saúde é importante para o Cliente 01, e ele busca dispositivos que ofereçam recursos avançados de saúde e bem-estar."
preferencia_4_client01 = "Cliente 01 prefere trabalhar com dispositivos que rodam o sistema operacional macOS e iOS, e valoriza a integração entre seus dispositivos Apple."

pedido_1_client02 = ["Samsung Galaxy S23 Ultra", "Carregador sem fio compatível com Samsung Galaxy", "Capa de Silicone para Samsung Galaxy S23 Ultra"]
pedido_2_client02 = ["Fone de Ouvido Samsung Galaxy Buds2 Pro", "Cabo USB-C para USB-C", "Suporte Veicular para Samsung Galaxy"]
pedido_3_client02 = ["Samsung Galaxy Watch5", "Pulseira Esportiva para Samsung Galaxy Watch5", "Carregador compatível com Samsung Galaxy Watch5"]
pedido_4_client02 = ["Samsung Galaxy Book3 Pro 16", "Adaptador USB-C para HDMI", "Bolsa para Samsung Galaxy Book3 Pro"]

preferencia_1_client02 = "Cliente 02 prefere produtos da Samsung e valoriza a qualidade e durabilidade dos dispositivos."
preferencia_2_client02 = "Tecnologia é uma prioridade para o Cliente 02, e ele está disposto a investir em produtos premium que ofereçam desempenho e inovação."
preferencia_3_client02 = "Monitoramento de saúde é importante para o Cliente 02, e ele busca dispositivos que ofereçam recursos avançados de saúde e bem-estar."
preferencia_4_client02 = "Cliente 02 prefere trabalhar com computadores Windows e smartphones Android com interface One UI, e valoriza a integração entre seus dispositivos Samsung."


In [ ]:
# Verificando o status inicial dos namespaces de pedidos e preferências
# Namespace de compras
resultados_busca_1 = semanticDB.search(ns_compras, query="Quais produtos mais populares no momento?", limit=1)
if not resultados_busca_1:
    print("Nenhum registro encontrado em compras")
else:
    for item in resultados_busca_1:
        print(f"ID: {item.key} | Conteúdo: {item.value}")

semanticDB.conn.commit()

# Namespace de preferências
resultados_busca_2 = semanticDB.search(ns_preferencias, query="Quais são as preferências comuns dos clientes?", limit=1)
if not resultados_busca_2:
    print("Nenhum registro encontrado em preferências")
else:
    for item in resultados_busca_2:
        print(f"ID: {item.key} | Conteúdo: {item.value}")

semanticDB.conn.commit()


In [ ]:
# Armazenando os pedidos dos Clientes 01 e 02 no namespace de compras

# Inserindo registros de pedidos para o Cliente 01 e Cliente 02 no namespace de compras
semanticDB.put(ns_compras, str(uuid.uuid4()), {"data": pedido_1_client01})
semanticDB.put(ns_compras, str(uuid.uuid4()), {"data": pedido_2_client01})
semanticDB.put(ns_compras, str(uuid.uuid4()), {"data": pedido_3_client01})
semanticDB.put(ns_compras, str(uuid.uuid4()), {"data": pedido_4_client01})
semanticDB.put(ns_compras, str(uuid.uuid4()), {"data": pedido_1_client02})
semanticDB.put(ns_compras, str(uuid.uuid4()), {"data": pedido_2_client02})
semanticDB.put(ns_compras, str(uuid.uuid4()), {"data": pedido_3_client02})
semanticDB.put(ns_compras, str(uuid.uuid4()), {"data": pedido_4_client02})

# Armazenando as preferências dos Clientes 01 e 02 no namespace de preferências
# Inserindo registros de preferências para o Cliente 01 e Cliente 02 no namespace de preferências
semanticDB.put(ns_preferencias, str(uuid.uuid4()), {"data": preferencia_1_client01})
semanticDB.put(ns_preferencias, str(uuid.uuid4()), {"data": preferencia_2_client01})
semanticDB.put(ns_preferencias, str(uuid.uuid4()), {"data": preferencia_3_client01})
semanticDB.put(ns_preferencias, str(uuid.uuid4()), {"data": preferencia_4_client01})
semanticDB.put(ns_preferencias, str(uuid.uuid4()), {"data": preferencia_1_client02})
semanticDB.put(ns_preferencias, str(uuid.uuid4()), {"data": preferencia_2_client02})
semanticDB.put(ns_preferencias, str(uuid.uuid4()), {"data": preferencia_3_client02})
semanticDB.put(ns_preferencias, str(uuid.uuid4()), {"data": preferencia_4_client02})



In [ ]:
#Executando consultas para verificar os dados armazenados no namespace preferências

resultados_busca = semanticDB.search(ns_preferencias, query="O que o Cliente 1 mais valoriza?", limit=1)

if not resultados_busca:
    print("Nenhum registro encontrado em preferências")
else:
    for item in resultados_busca:
        print(f"ID: {item.key} | Conteúdo: {item.value}")

semanticDB.conn.commit()

resultados_busca = semanticDB.search(ns_preferencias, query="Quais são as preferências do Cliente 2?", limit=1)

if not resultados_busca:
    print("Nenhum registro encontrado em preferências")
else:
    for item in resultados_busca:
        print(f"ID: {item.key} | Conteúdo: {item.value}")

semanticDB.conn.commit()




In [ ]:
#Executando consultas para verificar os dados armazenados no namespace preferências

resultados_busca = semanticDB.search(ns_compras, query="Que tipo de acessório o Cliente 1 gosta de comprar?", limit=1)

if not resultados_busca:
    print("Nenhum registro encontrado em preferências")
else:
    for item in resultados_busca:
        print(f"ID: {item.key} | Conteúdo: {item.value}")

semanticDB.conn.commit()

resultados_busca = semanticDB.search(ns_compras, query="Qual o fabricante o Cliente 2 mais comprou produtos?", limit=1)

if not resultados_busca:
    print("Nenhum registro encontrado em preferências")
else:
    for item in resultados_busca:
        print(f"ID: {item.key} | Conteúdo: {item.value}")

semanticDB.conn.commit()


In [ ]:
semanticDB.conn.close()